<a href="https://colab.research.google.com/github/donghong221201-jpg/Computer-Thinking-Week6/blob/main/lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11주차 실습: 난장판 폴더 확장자별 자동 청소하기

이 실습 노트북은 파일 확장자를 기준으로 자동 폴더 생성 및 이동을 수행하는 파이썬 스크립트를 작성하는 공간입니다. **코드를 외워서 직접 짜지 마세요.** 아래 단계에 따라 AI(Gemini)에게 요구하고 코드를 가져와서 에러를 해결해 보세요.

Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [ ]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

In [4]:
import os
import random

# Tạo thư mục 'downloads' nếu chưa có
folder_name = 'downloads'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

# Danh sách các phần mở rộng tệp giả định
extensions = ['jpg', 'png', 'pdf', 'docx', 'xlsx', 'txt']

# Tạo 100 tệp tin ngẫu nhiên
for i in range(100):
    ext = random.choice(extensions)
    file_name = f"test_file_{i}.{ext}"
    file_path = os.path.join(folder_name, file_name)

    # Tạo tệp rỗng
    with open(file_path, 'w') as f:
        f.write("Đây là nội dung tệp giả lập.")

print(f"Đã tạo xong 100 tệp tin rác trong thư mục '{folder_name}'.")

Đã tạo xong 100 tệp tin rác trong thư mục 'downloads'.


In [6]:
import os
import random

# --- CẤU HÌNH (Constants) ---
# Đặt tên biến viết hoa để báo hiệu đây là hằng số
DOWNLOADS_DIR = 'downloads'
TOTAL_FILES = 100
EXTENSIONS = ['jpg', 'png', 'pdf', 'docx', 'xlsx', 'txt']

def create_dummy_files(directory, count, extensions):
    """Hàm tạo tệp giả lập với tên được định dạng chuyên nghiệp."""
    # Tạo thư mục nếu chưa tồn tại
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"[*] Đã tạo thư mục: {directory}")

    for i in range(count):
        ext = random.choice(extensions)

        # Sử dụng :03d để tạo số thứ tự kiểu 000, 001, 002...
        # Điều này giúp tệp luôn được sắp xếp đúng thứ tự trong máy tính
        file_name = f"file_{i:03d}.{ext}"
        file_path = os.path.join(directory, file_name)

        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(f"Nội dung giả lập cho tệp {file_name}")

    print(f"[+] Thành công: Đã tạo {count} tệp tại '{directory}'.")

# Thực thi hàm
if __name__ == "__main__":
    create_dummy_files(DOWNLOADS_DIR, TOTAL_FILES, EXTENSIONS)

[+] Thành công: Đã tạo 100 tệp tại 'downloads'.


# Mục mới

## [1단계] 난장판(더미) 파일 준비하기
먼저, 청소할 가짜 쓰레기 파일들이 필요합니다. 코랩 환경 내에 임의의 파일 100개를 무작위로 생성하는 코드를 제미니에게 물어보고, 아래 셀에 붙여넣어 실행하세요.

* 💡 **프롬프트 팁**: "코랩 환경의 'downloads'라는 폴더 안에 임의의 가짜 파일 100개를 만들어주는 파이썬 코드를 짜줘. 확장자는 pdf, docx, jpg, png, txt, xlsx가 섞여 있게 해줘."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## [2단계] 확장자별 자동 분류 스크립트 작성
이제 위에서 만든 수많은 파일들을 확장자별로 새 폴더를 만들어 깔끔하게 이동시켜주는 분류 코드를 제미니에게 요청하고 아래 셀에 붙여넣어 실행하세요.

* 💡 **프롬프트 팁**: "'downloads' 폴더 안에 있는 파일들을 확장자별로 확인해서, 확장자 이름과 똑같은 새 폴더를 만들고 그 안으로 파일들을 모두 이동시켜주는 파이썬 코드를 만들어 줘. 이미 폴더가 있으면 만들지 말고 그냥 이동시켜 줘."

In [13]:
import os
import shutil

def classify_files_by_extension(source_dir):
    """
    Hàm này giúp dọn dẹp thư mục: tự động gom các tệp cùng loại vào một chỗ.
    """
    # Bước 1: Kiểm tra xem cái thư mục 'downloads' có trên máy không
    if not os.path.exists(source_dir):
        print(f"[!] Lỗi: Không tìm thấy thư mục '{source_dir}' để dọn dẹp.")
        return

    # Bước 2: Lấy danh sách tất cả những gì đang nằm trong thư mục đó
    all_items = os.listdir(source_dir)

    moved_count = 0
    for item in all_items:
        # Tạo đường dẫn đầy đủ đến tệp đó
        item_path = os.path.join(source_dir, item)

        # Bước 3: Chỉ xử lý nếu nó là 'tệp tin' (không xử lý nếu nó đã là thư mục)
        if os.path.isfile(item_path):
            # Bước 4: Tách lấy cái đuôi (extension) của tệp (ví dụ: .jpg, .pdf)
            name, ext = os.path.splitext(item)

            # Nếu có đuôi thì lấy chữ cái đuôi (bỏ dấu chấm), nếu không có thì gọi là 'no_extension'
            ext_folder = ext[1:].lower() if ext else 'no_extension'

            # Bước 5: Chuẩn bị một thư mục mới có tên trùng với cái đuôi đó
            target_dir = os.path.join(source_dir, ext_folder)

            # Nếu thư mục này chưa có thì máy tính tự tạo ra
            if not os.path.exists(target_dir):
                os.makedirs(target_dir)

            # Bước 6: 'Bốc' tệp tin thả vào trong thư mục mới tạo
            shutil.move(item_path, os.path.join(target_dir, item))
            moved_count += 1

    print(f"[+] Xong rồi! Đã dọn dẹp ngăn nắp {moved_count} tệp vào từng ô riêng.")

# Lệnh này để bắt đầu chạy dọn dẹp thư mục 'downloads'
if __name__ == '__main__':
    classify_files_by_extension('downloads')

[+] Xong rồi! Đã dọn dẹp ngăn nắp 0 tệp vào từng ô riêng.


## [3단계] 에러 해결 (Troubleshooting) ✨(가장 중요!)✨
코드를 실행하다가 `FileNotFoundError` 등 붉은색 에러가 났나요? 당황하지 마세요. 그것이 정상입니다!

에러 메시지를 복사해서 제미니에게 **"이 코드를 실행했더니 이런 에러가 나. 어떻게 고쳐야 해?"**라고 다시 물어보고, 해결된 코드를 다시 위 셀에 덮어써서 실행이 성공할 때까지 반복하세요.

**⚠️ 주의: 에러를 고치기 위해 제미니와 나눈 대화 기록은 이번 주 이메일 과제의 핵심 제출 내용입니다!**